In [12]:
from dotenv import load_dotenv
import os
load_dotenv()
import uuid
from typing import List
from pydantic import BaseModel,Field

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph,START, END,MessagesState
from langgraph.store.postgres import PostgresStore
from langgraph.store.base import BaseStore

In [2]:
SYSTEM_PROMPT_TEMPLATE="""
You are a helpful Assistant with memory capabilities.
If user Specific memory is available, use it to personalize your responses based on what you know about the user.

Your goal is to provide relevant, friendly and tailored assistance that refers the user's preferences, context and past interactions.

If the user's name or relevant personal context is available, always personalize your responses by:
Always address the user by name (e.g., "Sure, Maaz..") when appropriate
Referencing know projects, tools, or preferences (e.g., "Your MCP server python based project)
Adjusting the tone to feel friendly, natural and directly aimed at the user

Always generic phrasing when personalilizing is possible.

Use personalization especially in:
Greetings and transitions
Help or guidance tailored tools and frameworks the user uses 
Follow-up messages tht continue from past context

Always ensure that personalizations is based only on known user details and not assumed.
In the end suggest 3 relevant further questions based on the current response and user profile

The user's memory (which may be empty) is provided as : {user_details_content}
"""

In [3]:
memory_llm = ChatGroq(model="openai/gpt-oss-120b",max_tokens=600,temperature=0)

In [4]:
class MemoryItem(BaseModel):
    text: str=Field(description="Atomic user as a short sentence")
    is_new: bool = Field(description="True if this memory is new and should be stored. False if duplicates/already known")

class MemoryDecision(BaseModel):
    should_write: bool=Field(description="Whether to store any memories")
    memories: List[MemoryItem] = Field(default_factory=list, description="Atomic user memories to store")

In [5]:
memory_extractor = memory_llm.with_structured_output(MemoryDecision)

In [6]:
MEMORY_PROMPT= """
Your are responsible for updating and maintaining accurate user memories.
CURRENT USER DETAILS (Existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specifc info worth storing long0term (idetnity, stable preferences,ongoinf projeccts/goals).
- For each extracted item, set is_new=True Only if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false
- Keep each memory as a short by the user.
- If there is nothing memory-worthy, return an empty list.
"""

In [7]:
def remember_node(state: MessagesState,config: RunnableConfig, store: BaseStore):
    user_id=config["configurable"]["user_id"]

    namespace=("user",user_id,"details")

    items = store.search(namespace)
    existing = "\n".join(it.value['data'] for it in items) if items else "(empty)"


    last_msg = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(content=MEMORY_PROMPT.format(user_details_content=existing)),
                                   {"role":"user","content":last_msg},
        ]
    )

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new:
                store.put(namespace,str(uuid.uuid4()),{"data":mem.text})

    return {}

In [8]:
chat_llm = ChatGroq(model="openai/gpt-oss-120b",max_tokens=600,temperature=0)

In [9]:
def chat_node(state: MessagesState,config: RunnableConfig,store:BaseStore):
    user_id = config["configurable"]['user_id']

    namespace=("user",user_id,"details")

    items = store.search(namespace)
    user_details = "\n".join(it.value['data'] for it in items) if items else ""


    
    system_msg=SYSTEM_PROMPT_TEMPLATE.format(
        user_details_content=user_details or "(empty)"
    )

    response=chat_llm.invoke([system_msg] + state["messages"])
    return {"messages":[response]}

In [11]:
builder=StateGraph(MessagesState)
builder.add_node("remember",remember_node)
builder.add_node("chat",chat_node)

builder.add_edge(START,"remember")
builder.add_edge("remember","chat")

builder.add_edge("chat",END)



In [15]:
DB_URL= os.getenv("DB_URL")


with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    graph=builder.compile(store=store)

    config={"configurable":{"user_id":"u1"}}

    graph.invoke({"messages":[{"role":"user","content":"Hi my name is Maaz"}]},config)
    graph.invoke({"messages":[{"role":"user","content":"I am skilled in python "}]},config)
    out = graph.invoke({"messages":[{"role":"user","content":"Explain Gen AI in simple terms"}]},config)

    print(out["messages"][-1].content)

    print("\n---Stored memories----")
    for it in store.search(("user","u1","details")):
        print(it.value['data'])


Sure, Maaz! Let’s break down **generative AI** in a way that clicks with your Python background.

---

## What is Generative AI?

**Generative AI** is a type of artificial intelligence that can **create new content**—like text, images, music, or code—rather than just recognizing or classifying what already exists.

| Traditional AI | Generative AI |
|----------------|---------------|
| *What is this?* (e.g., “Is this a cat?”) | *Make something new* (e.g., “Write a short story about a cat.”) |
| Predictive, often classification‑oriented | Creative, often “sampling” from learned patterns |

### How does it work? (Python‑friendly view)

1. **Training Data** – You feed a huge dataset (e.g., millions of sentences, pictures, or code snippets) into a model.
2. **Neural Network** – Usually a deep model like a **Transformer** (think `torch.nn.Transformer` or `tensorflow.keras.layers.MultiHeadAttention`).
3. **Learning the Distribution** – The model learns the statistical patterns of the data, e

User is skilled in Python.
User's name is Maaz
